# EIS Collection Analysis

Load collected Parquet tables from `data/processed/` and summarize the
attachments and attribute coverage.

In [1]:
from pathlib import Path
import re
import pandas as pd


def resolve_processed_base() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data" / "processed",
        cwd / "processed",
    ]
    for parent in [cwd, *cwd.parents]:
        candidates.append(parent / "data" / "processed")
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not locate data/processed directory from cwd")


def extract_run_id(path: Path) -> int | None:
    match = re.search(r"run_(\d+)_", path.name)
    if not match:
        return None
    return int(match.group(1))


def latest_run_id(path: Path) -> int | None:
    run_ids = [extract_run_id(p) for p in path.glob("*.parquet")]
    run_ids = [rid for rid in run_ids if rid is not None]
    return max(run_ids) if run_ids else None


def read_run_parquets(path: Path, run_id: int | None) -> pd.DataFrame:
    if run_id is None:
        files = list(path.glob("*.parquet"))
    else:
        files = list(path.glob(f"*run_{run_id}_*.parquet"))
    if not files:
        return pd.DataFrame()
    frames = [pd.read_parquet(file) for file in sorted(files)]
    return pd.concat(frames, ignore_index=True)


base = resolve_processed_base()
print("Using processed base:", base)

run_id = latest_run_id(base / "guarantees")
print("Latest run_id:", run_id)

guarantees = read_run_parquets(base / "guarantees", run_id)
attributes = read_run_parquets(base / "attributes", run_id)
files = read_run_parquets(base / "files", run_id)

print("guarantees rows", len(guarantees))
print("attributes rows", len(attributes))
print("files rows", len(files))

display(guarantees.head())
display(attributes.head())
display(files.head())

Using processed base: /Users/home/Work/10-edu/data-science/thesis/code/masters-thesis-dev/data/processed
Latest run_id: 62
guarantees rows 2
attributes rows 105
files rows 2


,run_id,id,status,general_url,documents_url,fetched_at,warnings,error
0,62,1750000,OK,https://zakupki.gov.ru/epz/bankguarantee/guara...,https://zakupki.gov.ru/epz/bankguarantee/guara...,2026-02-26T05:12:10.980961+00:00,[],
1,62,1750001,OK,https://zakupki.gov.ru/epz/bankguarantee/guara...,https://zakupki.gov.ru/epz/bankguarantee/guara...,2026-02-26T05:12:13.924755+00:00,[],


,run_id,id,section,field_name,field_value,document_index,document_number
0,62,1750000,Сводная информация (верхний блок),Статус,Размещено,NaN,NaN
1,62,1750000,Сводная информация (верхний блок),Банк-гарант,"ПУБЛИЧНОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО ""БИНБАНК""",NaN,NaN
2,62,1750000,Сводная информация (верхний блок),ИНН,5408117935,NaN,NaN
3,62,1750000,Сводная информация (верхний блок),КПП,770501001,NaN,NaN
4,62,1750000,Сводная информация (верхний блок),Номер извещения об осуществлении закупки,0338200009818000013,NaN,NaN


,run_id,id,file_index,stored_filename,stored_path,original_filename,download_url,document_index,document_number,page_count,mime_type,download_status,sha256
0,62,1750000,1,1750000_1.pdf,/Users/home/Work/10-edu/data-science/thesis/co...,18777-447-163052.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,1,00O2410103915318002601,2,application/pdf,DOWNLOADED,0014a05b1da46e40bad6f2fa37126fba69b9752be74a7f...
1,62,1750001,1,1750001_1.pdf,/Users/home/Work/10-edu/data-science/thesis/co...,БГ 974 УМПК.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,1,0L61027402236618000801,2,application/pdf,DOWNLOADED,684af49ddbe080a17c6ef60bb5769bdc282eaeada88367...


In [2]:
# summary for the last run
if files.empty:
    print("No files table found.")
else:
    files = files.copy()
    files["extension"] = files["stored_filename"].str.extract(r"(\.[^\.]+)$", expand=False).fillna("")

    def file_size(path: str) -> int:
        try:
            return Path(path).stat().st_size
        except FileNotFoundError:
            return 0
        except OSError:
            return 0

    files["size_bytes"] = files["stored_path"].apply(file_size)

    print("Download status counts")
    display(files["download_status"].value_counts())

    print("File extensions")
    display(files["extension"].value_counts())

    total_size = int(files["size_bytes"].sum())
    downloaded_size = int(files.loc[files["download_status"].eq("DOWNLOADED"), "size_bytes"].sum())

    print("Total size (bytes)", total_size)
    print("Downloaded size (bytes)", downloaded_size)
    print("Existing files", int((files["size_bytes"] > 0).sum()))

    if "sha256" in files.columns:
        nonempty = files[files["sha256"].astype(str).str.len() > 0]
        dupes = nonempty[nonempty.duplicated("sha256", keep=False)]
        if dupes.empty:
            print("No duplicate hashes detected.")
        else:
            print("Duplicate hashes detected")
            display(dupes.sort_values("sha256"))

Download status counts


download_status
DOWNLOADED    2
Name: count, dtype: int64

File extensions


extension
.pdf    2
Name: count, dtype: int64

Total size (bytes) 1272765
Downloaded size (bytes) 1272765
Existing files 2
No duplicate hashes detected.


In [3]:
# attributes['field_name'].value_counts()

In [4]:
# guarantees.head()

In [5]:
from pathlib import Path
import hashlib


def read_all_parquets(path: Path) -> pd.DataFrame:
    files = list(path.glob("*.parquet"))
    if not files:
        return pd.DataFrame()
    frames = [pd.read_parquet(file) for file in sorted(files)]
    return pd.concat(frames, ignore_index=True)


guarantees_all = read_all_parquets(base / "guarantees")
attributes_all = read_all_parquets(base / "attributes")
files_all = read_all_parquets(base / "files")

if guarantees_all.empty:
    raise RuntimeError("No guarantees tables found in processed folder.")

if "run_id" not in guarantees_all.columns:
    raise RuntimeError("guarantees table missing run_id; rerun parser with run_id support.")

# Pick the latest run per guarantee ID.
guarantees_latest = (
    guarantees_all.sort_values(["id", "run_id", "fetched_at"]).drop_duplicates("id", keep="last")
)

latest_keys = guarantees_latest[["id", "run_id"]].drop_duplicates()
attributes_latest = attributes_all.merge(latest_keys, on=["id", "run_id"], how="inner")
files_latest = files_all.merge(latest_keys, on=["id", "run_id"], how="inner")


def file_exists(path_value: str) -> bool:
    if not isinstance(path_value, str) or not path_value:
        return False
    return Path(path_value).exists()


files_latest["file_exists"] = files_latest["stored_path"].apply(file_exists)
files_latest_existing = files_latest[files_latest["file_exists"]].copy()

print("Latest guarantees rows", len(guarantees_latest))
print("Latest attributes rows", len(attributes_latest))
print("Latest files rows", len(files_latest))
print("Existing files rows", len(files_latest_existing))

Latest guarantees rows 571079
Latest attributes rows 28376493
Latest files rows 594966
Existing files rows 593388


In [6]:
# these files doesn't exist
files_latest[files_latest['file_exists'] == False]

,id,file_index,stored_filename,stored_path,original_filename,download_url,mime_type,download_status,sha256,document_index,document_number,page_count,run_id,file_exists
48,461122,1,461122_1.pdf,/Users/home/Work/10-edu/data-science/thesis/co...,11122015193909_002.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,,FAILED_TIMEOUT,,1.0,01Q2772339668515000501,0.0,NaN,False
69,119178,1,,,1941.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,,FAILED_NOT_FOUND,,1.0,0420173200004614004101,0.0,14.0,False
132,40710,2,,,Договор БГ 385-53.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,,FAILED_NOT_FOUND,,1.0,05A0141200001514000603,0.0,17.0,False
861,15,1,,,"БГ 018 (01814, 15 млн).jpg",https://zakupki.gov.ru/44fz/filestore/public/1...,,FAILED_NOT_FOUND,,1.0,0930373100042414000101,0.0,28.0,False
880,66,2,,,гарантия 14103.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,,FAILED_NOT_FOUND,,2.0,0890373200001614000101,0.0,28.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
585383,1453885,1,,,Оригинал БГ 026275.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,,FAILED_TIMEOUT,,1.0,04E3141326190517000101,0.0,61.0,False
586908,1453886,1,,,11152017101955_004.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,,FAILED_TIMEOUT,,1.0,0J52143502651017010201,0.0,61.0,False
591399,1452934,1,,,scan_010.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,,FAILED_TIMEOUT,,1.0,0J53380113206817002601,0.0,61.0,False
592918,1455367,1,,,Scan 001.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,,FAILED_TIMEOUT,,1.0,02J2132512662517013501,0.0,61.0,False


In [7]:
# sample
files_latest_existing[files_latest_existing['id'] == 40710]

,id,file_index,stored_filename,stored_path,original_filename,download_url,mime_type,download_status,sha256,document_index,document_number,page_count,run_id,file_exists
131,40710,1,40710_1.pdf,/Users/home/Work/10-edu/data-science/thesis/co...,Договор БГ 385-53.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,application/pdf,DOWNLOADED,e829cda7956f706ea1424c8cd784bf0ea94906553a1708...,1.0,05A0141200001514000603,3.0,17.0,True
133,40710,3,40710_3.pdf,/Users/home/Work/10-edu/data-science/thesis/co...,Договор БГ 385-53.pdf,https://zakupki.gov.ru/44fz/filestore/public/1...,application/pdf,DOWNLOADED,e829cda7956f706ea1424c8cd784bf0ea94906553a1708...,2.0,05A0141200001514000601,3.0,17.0,True


In [8]:
final_dir = base / "final"
final_dir.mkdir(parents=True, exist_ok=True)

guarantees_latest.to_csv(final_dir / "guarantees_latest.csv", index=False)
attributes_latest.to_csv(final_dir / "attributes_latest.csv", index=False)
files_latest_existing.to_csv(final_dir / "files_latest.csv", index=False)

print("Saved CSVs to", final_dir)

Saved CSVs to /Users/home/Work/10-edu/data-science/thesis/code/masters-thesis-dev/data/processed/final


In [9]:
def compute_sha256(path: Path) -> str:
    hasher = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


total_files = len(files_latest_existing)
sha_series = files_latest_existing.get("sha256")
if sha_series is None:
    sha_series = pd.Series([""] * total_files)

sha_series = sha_series.fillna("")
missing_sha = files_latest_existing[sha_series.eq("")]

if not missing_sha.empty:
    computed = []
    for path_value in missing_sha["stored_path"]:
        try:
            computed.append(compute_sha256(Path(path_value)))
        except Exception:
            computed.append("")
    sha_series.loc[missing_sha.index] = computed

unique_files = sha_series[sha_series.ne("")].nunique()

print("Total downloaded files (existing):", total_files)
print("Unique files by sha256:", unique_files)

Total downloaded files (existing): 593388
Unique files by sha256: 532075


In [10]:
# files_latest_existing[files_latest_existing['stored_filename'].str.endswith(".jpg")]

In [11]:
# files_latest_existing[files_latest_existing['stored_filename'].str.endswith(".doc")].head()

### Draw random samples

In [12]:
import numpy as np
np.random.seed(13)
print(np.random.choice(list(range(1, 1962721)), 10), sep=',')

[1540435 1751217 1015883  253457  256743 1737571 1791733 1900698 1860333
 1216443]


In [13]:
import random

# Generate 25 random integers between 1 and 2000000
random_numbers = [str(random.randint(1, 1962721)) for _ in range(1000)]

# Print as a comma-separated string
print(",".join(random_numbers))

1299038,721194,1934392,558990,1269057,1411705,1480909,766640,883117,144688,84069,972305,218179,1550362,1356836,1733091,1690514,877819,973293,538159,1786990,840582,962251,718260,1408140,739036,570977,1849329,1668364,1795502,745431,44687,864394,1607338,526206,1432302,800372,523020,1637674,948584,816790,1021370,211214,320256,1027170,1707896,85922,1020307,371178,1584853,1658516,814997,1767848,209263,1767574,464481,569474,1921370,1130967,777520,1908379,1711135,716720,903051,1770036,1091724,939426,1207000,908394,42683,492049,1891149,584985,1362045,1152383,1164715,627169,748895,640050,1777415,953777,332412,169295,1873430,127817,84278,702217,6711,248736,1543218,1715949,327352,688578,1529136,1408895,1899225,1593506,1170379,1353604,449029,676377,1950196,1053942,1137034,411055,356904,1681770,156150,195670,1385037,951273,80813,1102462,1326655,1799617,1613808,432314,692373,1741091,627278,798965,1737846,473856,138464,1864968,226588,1544643,986931,1153026,1001899,1266962,921214,521732,1356105,1782430